# INUA Sprint 2: Exploratory Data Analysis (EDA)

**Student:** Michelle  
**Project:** INUA - Intelligent Mentorship Platform  
**Institution:** Strathmore University (Final Year Project - BICS Year 4)  
**Deliverables:** Dissertation Chapter 4 Tables (4.1-4.3) & Figures (4.1-4.7)  

---

## Objectives of this EDA Sprint
1. **Data Integrity & Completeness**: Confirm zero missingness across all 2,000 synthetic interaction records.
2. **Univariate Distributions**: Inspect mentee cohorts, faculty representation, and session typologies.
3. **Hypothesis & Rule Validation**: Verify empirical relationships (Mentor experience vs Career support, Communication vs Psychosocial support).
4. **Multivariate Correlation Analysis**: Generate the psychometric correlation matrix heatmap.
5. **Target Imbalance & Yield Inspection**: Analyze `session_effectiveness` across attendance modalities.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Publication Styling Configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.labelweight': 'semibold',
    'figure.titlesize': 14,
})

DATA_PATH = os.path.join('..', 'data', 'inua_mentorship_dataset.csv')
FIG_DIR = os.path.join('..', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Missing Values & Summary Statistics (Table 4.1 & 4.2)

In [ ]:
# Table 4.1: Missing Values and Data Types
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.astype(str),
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
}).reset_index(drop=True)

print("Total null entries across entire dataset:", df.isnull().sum().sum())
missing_df

In [ ]:
# Table 4.2: Parametric Summary Statistics for Numerical & Likert Features
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols].describe().T.round(2)

## 2. Demographic Analysis (Figures 4.1 & 4.2)
- **Figure 4.1**: Career Stage Distribution
- **Figure 4.2**: Field of Study Representation

In [ ]:
# Figure 4.1: Career Stage Distribution
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
stage_order = ['Year1', 'Year2', 'Year3', 'Year4', 'Graduate', 'EarlyCareer']
stage_counts = df['career_stage'].value_counts().reindex(stage_order)

bars = ax.bar(stage_order, stage_counts, color='#6D28D9', edgecolor='#4C1D95', width=0.6)
for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(df)) * 100
    ax.text(bar.get_x() + bar.get_width()/2, yval + 10, f"{yval}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title("Figure 4.1: Mentee Distribution by Academic Career Stage")
ax.set_xlabel("Career Stage")
ax.set_ylabel("Number of Sessions")
ax.set_ylim(0, max(stage_counts) * 1.25)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 4.2: Field of Study Distribution
fig, ax = plt.subplots(figsize=(9, 4.8), dpi=150)
field_counts = df['field_of_study'].value_counts(ascending=True)

bars = ax.barh(field_counts.index, field_counts, color='#8B5CF6', edgecolor='#6D28D9')
for bar in bars:
    xval = bar.get_width()
    pct = (xval / len(df)) * 100
    ax.text(xval + 8, bar.get_y() + bar.get_height()/2, f"{xval} ({pct:.1f}%)", ha='left', va='center', fontsize=9, fontweight='bold')

ax.set_title("Figure 4.2: Mentee Distribution Across Undergraduate Disciplines")
ax.set_xlabel("Number of Sessions")
ax.set_ylabel("Field of Study")
ax.set_xlim(0, max(field_counts) * 1.25)
plt.tight_layout()
plt.show()

## 3. Session Typology & Duration Dynamics (Figures 4.3 & 4.5)
- **Figure 4.3**: Session Type Allocation
- **Figure 4.5**: Empirical Session Duration (Minutes) Conditioned on Session Type

In [ ]:
# Figure 4.3: Session Type Allocation
fig, ax = plt.subplots(figsize=(8.5, 4.5), dpi=150)
st_counts = df['session_type'].value_counts()

bars = ax.bar(st_counts.index, st_counts, color='#4338CA', edgecolor='#312E81', width=0.55)
for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(df)) * 100
    ax.text(bar.get_x() + bar.get_width()/2, yval + 10, f"{yval}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title("Figure 4.3: Proportional Allocation of Mentorship Session Types")
ax.set_xlabel("Session Purpose")
ax.set_ylabel("Total Scheduled Sessions")
ax.set_ylim(0, max(st_counts) * 1.20)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 4.5: Duration by Session Type (Validates Rule 2)
fig, ax = plt.subplots(figsize=(9, 4.8), dpi=150)
order = ['Networking', 'CV Review', 'Career Guidance', 'Interview Preparation', 'Technical Coaching']

sns.violinplot(
    data=df, x='session_type', y='session_duration_minutes', order=order,
    palette='crest', inner='quartile', cut=0, ax=ax, hue='session_type', legend=False
)
ax.set_title("Figure 4.5: Empirical Duration Distributions Conditioned on Session Type")
ax.set_xlabel("Session Type")
ax.set_ylabel("Duration (Minutes)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Theory & Hypothesis Testing (Figures 4.4 & 4.6)
- **Figure 4.4**: Mentor Experience vs Career Support (Rule 3 logarithmic validation)
- **Figure 4.6**: Correlation Matrix Heatmap across Likert indices and composite effectiveness

In [ ]:
# Figure 4.4: Mentor Seniority vs Career Support
fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=150)
df_copy = df.copy()
bins = [0, 2, 5, 9, 15, 30]
labels = ['1-2 yrs', '3-5 yrs', '6-9 yrs', '10-15 yrs', '16+ yrs']
df_copy['exp_bracket'] = pd.cut(df_copy['mentor_experience_years'], bins=bins, labels=labels)

sns.boxplot(
    data=df_copy, x='exp_bracket', y='career_support_score', ax=ax,
    palette='Purples', boxprops=dict(alpha=0.85), showmeans=True,
    meanprops={'marker': 'D', 'markerfacecolor': '#EF4444', 'markeredgecolor': 'black', 'markersize': 6},
    hue='exp_bracket', legend=False
)

means = df_copy.groupby('exp_bracket', observed=False)['career_support_score'].mean()
for idx, (lbl, mean_val) in enumerate(means.items()):
    ax.text(idx, mean_val + 0.18, f"Mean: {mean_val:.2f}", ha='center', va='bottom', fontsize=9, fontweight='bold', color='#991B1B')

ax.set_title("Figure 4.4: Career Support Likert Score Across Mentor Seniority Brackets")
ax.set_xlabel("Mentor Experience Bracket (Years)")
ax.set_ylabel("Career Support Rating (1-5)")
ax.set_ylim(0.5, 5.8)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 4.6: Pearson Correlation Heatmap
fig, ax = plt.subplots(figsize=(8.5, 6.8), dpi=150)
cols = [
    'career_support_score', 'psychosocial_support_score', 'communication_quality',
    'goal_progress_score', 'confidence_growth_score', 'mentor_responsiveness_score',
    'effectiveness_index'
]
col_labels = [
    'Career Support', 'Psychosocial Support', 'Communication Quality',
    'Goal Progress', 'Confidence Growth', 'Mentor Responsiveness', 'Effectiveness Index'
]

corr_matrix = df[cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='Purples', vmin=-0.1, vmax=1.0,
    square=True, xticklabels=col_labels, yticklabels=col_labels, ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation Coefficient (r)'}
)

ax.set_title("Figure 4.6: Correlation Matrix of Psychometric Mentorship Features")
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Target Variable Analysis (Figure 4.7)
Ground-truth binary label `session_effectiveness` overall class split and yield conditional on `attendance_status`.

In [ ]:
# Figure 4.7: Ground-Truth Mentorship Effectiveness Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)

target_counts = df['session_effectiveness'].value_counts()
ax1.pie(
    [target_counts[1], target_counts[0]],
    labels=['Effective (1)', 'Ineffective/Disrupted (0)'],
    autopct='%1.1f%%', startangle=140, colors=['#10B981', '#EF4444'],
    explode=(0.04, 0), textprops={'fontweight': 'bold'}
)
ax1.set_title("Overall Ground-Truth Class Balance")

att_cross = pd.crosstab(df['attendance_status'], df['session_effectiveness'], normalize='index') * 100
att_cross.plot(kind='bar', stacked=True, color=['#EF4444', '#10B981'], ax=ax2, edgecolor='black', alpha=0.9)
ax2.set_title("Effectiveness Yield Conditioned on Attendance")
ax2.set_xlabel("Attendance Status")
ax2.set_ylabel("Proportion of Sessions (%)")
ax2.legend(['Ineffective (0)', 'Effective (1)'], loc='upper left', frameon=True)
plt.xticks(rotation=0)

fig.suptitle("Figure 4.7: Ground-Truth Mentorship Effectiveness Distribution", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()